In [2]:
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(Biostrings)
    library(data.table)
    library(igraph)
    library(pwalign)
    library(Rsamtools)
    library(parallel)
})

source("/srv/home/mlef0011/VDARK/src/utils.r")

WD       <- "/srv/home/mlef0011/VDARK"
MINIMAP  <- "/srv/home/mlef0011/anaconda3/envs/VDARK/bin/minimap2"
REF      <- "/srv/home/mlef0011/rawdata/ref_genome/reference_genome_GRCh37.fa"
N_CORES  <- max(1, detectCores() - 1)

### Load data

In [3]:

load_reads <- function(r1, r2) {
    fq1 <- readDNAStringSet(r1, format = "fastq")
    fq2 <- readDNAStringSet(r2, format = "fastq")
    data.frame(
        read_ID  = c(names(fq1), names(fq2)),
        sequence = as.character(c(fq1, fq2)),
        stringsAsFactors = FALSE
    )
}

tumour_reads <- load_reads(
    file.path(WD, "rawdata/reads/tumour_R1_tumour_specific.fq"),
    file.path(WD, "rawdata/reads/tumour_R2_tumour_specific.fq")
)

### Associate each k-mer with its original read.

In [4]:
system("/srv/home/mlef0011/anaconda3/condabin/conda run -n VDARK python3 /srv/home/mlef0011/VDARK/src/build_read_kmer_table.py")

### Clustering reads sharing same k-mers to perfom local assembly

In [5]:
read_kmer_df <- fread(file.path(WD, "rawdata/reads/read_kmer_association.tsv"))
read_kmer_df <- read_kmer_df[, if (.N <= 50) .SD, by = kmer]   # drop high-freq k-mers

pairs <- read_kmer_df[, {
    ids <- read_ID
    if (length(ids) >= 2) {
        idx <- combn(length(ids), 2)
        data.table(read_ID.x = ids[idx[1,]], read_ID.y = ids[idx[2,]])
    }
}, by = kmer][, .(weight = .N), by = .(read_ID.x, read_ID.y)]

G <- graph_from_data_frame(pairs, directed = FALSE)
E(G)$weight <- pairs$weight
clusters <- components(G)
cat(clusters$no, " clusters")

3235  clusters

In [19]:
cluster_ID = 1

### Construct tumor contig

In [20]:
reads_in  <- names(clusters$membership)[clusters$membership == cluster_ID]
kmers_sig <- unique(read_kmer_df$kmer[read_kmer_df$read_ID %in% reads_in])
all_kmers <- unique(unlist(lapply(
    tumour_reads$sequence[tumour_reads$read_ID %in% reads_in], get_kmers
)))

cat("  ", length(reads_in), " reads | ", length(kmers_sig), " specific k-mers")

contig_tumour <- assemble_kmers(kmers_sig, k = K)
cat("  Tumour contig: ", nchar(contig_tumour[1]), " bp")
contig_tumour

   20  reads |  53  specific k-mers  Tumour contig:  61  bp

[1] "AGCTCTCTAAGACTGCACTACAACCTGTGTAATCCCTATTGCTTCCAGAAATATAATGATG"
[2] "CATCATTATATTTCTGGAAGCAATAGGGATTACACAGGTTGTAGTGCAGTCTTAGAGAGCT"

### Fetch normal reads via flanking k-mers 


In [21]:
flanking_kmers <- setdiff(all_kmers, kmers_sig)
flanking_kmers <- flanking_kmers[grepl("^[ACGT]+$", flanking_kmers)]
flanking_fa    <- file.path(WD, "tmp", paste0("flanking_", cluster_ID, ".fa"))
kmc_db         <- file.path(WD, "tmp", paste0("flanking_kmc_", cluster_ID))

writeLines(paste0(">k_", seq_along(flanking_kmers), "\n", flanking_kmers), flanking_fa)

system(paste(KMC, "-k31 -t12 -ci1 -fm", flanking_fa, kmc_db, file.path(WD, "tmp")),
       ignore.stdout = TRUE, ignore.stderr = TRUE)

fq_R1_out <- file.path(WD, "tmp", paste0("normal_R1_locus_", cluster_ID, ".fq"))
fq_R2_out <- file.path(WD, "tmp", paste0("normal_R2_locus_", cluster_ID, ".fq"))

NORMAL_R1 = file.path(WD,"rawdata/reads/normal_R1.fq")
NORMAL_R2 = file.path(WD,"rawdata/reads/normal_R2.fq")

system(paste(KMC_TOOLS, "filter", kmc_db, "-ci1", NORMAL_R1, fq_R1_out),
       ignore.stdout = TRUE, ignore.stderr = TRUE)
system(paste(KMC_TOOLS, "filter", kmc_db, "-ci1", NORMAL_R2, fq_R2_out),
       ignore.stdout = TRUE, ignore.stderr = TRUE)

normal_df <- tryCatch(
    load_reads(fq_R1_out, fq_R2_out),
    error = function(e) data.frame(read_ID = character(), sequence = character())
)

In [23]:
head(normal_df)

,read_ID,sequence
,<chr>,<chr>
1,D25MWACXX130521:7:1104:17518:15497,TTAACATAAATTTCTGTACAAATTTGATTATTTCCTTAGGATAATTTCATCAAAAAAATTTGGTGCCTGGTCTCTGGGTCCAGCTGCTGGGAAGGAGGACA
2,D25MWACXX130521:7:2115:21209:32392,TGCAAAAATAAAGCAAAATGATAACTACCTAAAAGCTGTTCAGCTCTCTAAGACTGCACTACAACCTGTGTGATCCCTATTGCTTCCAGAAATATAATGAT
3,D25MWACXX130521:7:1308:16638:73870,AGCATAGTTAATCAGCAGCTTGCACTTCTGAGGGCACATTCACCTCGGGGGAGGAGGAAGGAGGAGGAGCTGGTGGGTGCTGAGACTGCAGTGTCCTCCTT
4,D25MWACXX130521:8:2308:7038:67266,ACTACCTAAAAGCTGTTCAGCTCTCTAAGACTGCACTACAACCTGTGTGATCCCTATTGCTTCCAGAAATATAATGATGGTTTTGCCTTTTTATTTTGATT
5,D25MWACXX130521:8:2109:4530:28732,AAAGAAACATGTTTGTCATTTTCCAAGAAAGAAAAGTAAAATCTAATCTACTCGACTTTTATGAATTTGTGAATTAATGCAAAAATAAAGCAAAATGATAA
6,D25MWACXX130521:7:1114:4941:89508,ATAAAGCAAAATGATAACTACCTAAAAGCTGTTCAGCTCTCTAAGACTGCACTACAACCTGTGTGATCCCTATTGCTTCCAGAAATATAATGATGGTTTTG


In [24]:
# Precise filtering: keep reads with ≥ 30 flanking k-mer hits

if (nrow(normal_df) > 0) {
    pdict  <- PDict(unique(c(flanking_kmers,
                             as.character(reverseComplement(DNAStringSet(flanking_kmers))))))
    n_hits <- vapply(vwhichPDict(pdict, DNAStringSet(normal_df$sequence)),
                     length, integer(1))
    normal_df <- normal_df[n_hits >= 30, ]
}

if (nrow(normal_df) == 0) {
    log_msg("  No normal reads at locus — skipping")
    return(NULL)
}

In [25]:
head(normal_df)

,read_ID,sequence
,<chr>,<chr>
2,D25MWACXX130521:7:2115:21209:32392,TGCAAAAATAAAGCAAAATGATAACTACCTAAAAGCTGTTCAGCTCTCTAAGACTGCACTACAACCTGTGTGATCCCTATTGCTTCCAGAAATATAATGAT
3,D25MWACXX130521:7:1308:16638:73870,AGCATAGTTAATCAGCAGCTTGCACTTCTGAGGGCACATTCACCTCGGGGGAGGAGGAAGGAGGAGGAGCTGGTGGGTGCTGAGACTGCAGTGTCCTCCTT
4,D25MWACXX130521:8:2308:7038:67266,ACTACCTAAAAGCTGTTCAGCTCTCTAAGACTGCACTACAACCTGTGTGATCCCTATTGCTTCCAGAAATATAATGATGGTTTTGCCTTTTTATTTTGATT
6,D25MWACXX130521:7:1114:4941:89508,ATAAAGCAAAATGATAACTACCTAAAAGCTGTTCAGCTCTCTAAGACTGCACTACAACCTGTGTGATCCCTATTGCTTCCAGAAATATAATGATGGTTTTG
7,D25MWACXX130521:8:1302:10258:62843,AAAGCAAAATGATAACTACCTAAAAGCTGTTCAGCTCTCTAAGACTGCACTACAACCTGTGTGATCCCTATTGCTTCCAGAAATATAATGATGGTTTTGCC
9,D25MWACXX130521:8:1207:12791:99793,CAATAGGGATCACACAGGTTGTAGTGCAGTCTTAGAGAGCTGAACAGCTTTTAGGTAGTTATCATTTTGCTTTATTTTTGCATTAATTCACAAATTCATAA


### Assemble normal contig

In [16]:
names(kmer_counts[kmer_counts >= 10])

character(0)

In [15]:
kmer_counts   <- table(unlist(lapply(normal_df$sequence, get_kmers)))
kmer_to_assemble = names(kmer_counts[kmer_counts >= 10])
if (length(kmer_to_assemble) == 0) {
    log_msg("  No normal reads at locus — skipping")
    return(NULL)
}
contig_normal <- assemble_kmers(kmer_to_assemble, k = K)

contig_normal
cat("Contig normal :", nchar(contig_normal[1]), "bp\n")


AAAAAAAGTATGTAACATATATTTATTTTAT AAAAAAGTATGTAACATATATTTATTTTATT 
                              2                               2 
AAAAACAAATGTATATAGCATATATTTATTT AAAAACACAATGTATATAACATATATTTAAT 
                              5                               6 
AAAAACATAAATGTATATAACATATTTAAGA AAAAACATAAATGTATATAACATATTTATTT 
                              1                               6 
AAAAACATAAATGTATATAACATTTATTTTA AAAAAGTATGTAACATATATTTATTTTATTT 
                              2                               2 
AAAACAAATGTATATAGCATATATTTATTTT AAAACACAATGTATATAACATATATTTAATT 
                              5                               7 
AAAACATAAATGTATATAACATATTTAAGAT AAAACATAAATGTATATAACATATTTATTTT 
                              1                               5 
AAAACATAAATGTATATAACATTTATTTTAT AAAAGTATGTAACATATATTTATTTTATTTA 
                              2                               3 
AAAATAAATATATGTTACATACTTTTTTTAT AAAATAAATATATGTTACATAGATCGGAAGA 
                        

NULL

Contig normal :  bp


In [ ]:
all_kmers_normal <- unlist(lapply(
    normal_reads_df$sequence[normal_reads_df$read_ID %in% normal_reads_locus], get_kmers))
kmer_counts   <- table(all_kmers_normal)
contig_normal <- assemble_kmers(names(kmer_counts[kmer_counts >= 10]), k = K)
contig_normal
cat("Contig normal :", nchar(contig_normal[1]), "bp\n")

### Alignment to retrieve SNV

In [13]:
# Meilleur alignement parmi les 4 orientations
alns <- lapply(list(
    c(contig_normal[1], contig_tumour[1]),
    c(contig_normal[1], contig_tumour[2]),
    c(contig_normal[2], contig_tumour[1]),
    c(contig_normal[2], contig_tumour[2])
), function(x) pairwiseAlignment(
    DNAString(x[1]), DNAString(x[2]), type = "local",
    substitutionMatrix = nucleotideSubstitutionMatrix(match=1, mismatch=-1),
    gapOpening = -5, gapExtension = -2
))
best_aln <- alns[[which.max(sapply(alns, score))]]

# Visualisation
pat  <- strsplit(as.character(pattern(best_aln)), "")[[1]]
sub  <- strsplit(as.character(subject(best_aln)),  "")[[1]]
diff <- ifelse(pat == sub, ".", "*")
cat("Normal :", paste(pat,  collapse=""), "\n")
cat("        ", paste(diff, collapse=""), "\n")
cat("Tumeur :", paste(sub,  collapse=""), "\n")

# SNVs
mm <- mismatchTable(best_aln)

if (nrow(mm) == 0) { cat("Aucune mutation détecté\n"); return(NULL) }
cat(nrow(mm), "Candidate SNV(s) :\n")
for (i in seq_len(nrow(mm))) {
    cat("  ", as.character(mm$PatternSubstring[i]), "->",
              as.character(mm$SubjectSubstring[i]),
        "| position dans contig :", mm$PatternStart[i], "\n")
}

Normal : GGTCCAGTGGCTTCAGATCCCTGGGTTATGGAGGCTGTGGCTTCCCTTCCCTGGGCTATGG 
         ..............................*.............................. 
Tumeur : GGTCCAGTGGCTTCAGATCCCTGGGTTATGAAGGCTGTGGCTTCCCTTCCCTGGGCTATGG 
1 Candidate SNV(s) :
   G -> A | position dans contig : 116 


In [15]:
germline <- is_germline_fast(mm, contig_normal)
mm_somatic <- mm[!germline, ]
mm_somatic

,PatternId,PatternStart,PatternEnd,PatternSubstring,SubjectStart,SubjectEnd,SubjectSubstring
,<int>,<int>,<int>,<chr>,<int>,<int>,<chr>
1,1,116,116,G,31,31,A


In [88]:
writeLines(c(">contig_normal1", contig_normal[1]), 
           "/srv/home/mlef0011/VDARK/rawdata/contigs/contigs_normal.fa")

In [90]:
# --- Lire le SAM ---
sam <- read.table("/srv/home/mlef0011/VDARK/rawdata/contigs/contigs_normal_aln.sam", 
                  sep="\t", comment.char="@", fill=TRUE)
# Flag du read
flag <- sam[1, 2]
is_reverse <- bitwAnd(flag, 16) != 0  # TRUE si brin reverse
is_reverse
chrom <- sam[1, 3]
pos   <- sam[1, 4]  # position de début du contig sur le génome
contig = sam[1, 10]
# --- Boucle sur les mutations détectées ---
# Parser le soft-clip depuis le CIGAR
cigar <- as.character(sam[1, 6])

# Soft-clip gauche (S) ou hard-clip gauche (H)
left_clip <- 0
m <- regmatches(cigar, regexpr("^([0-9]+)[SH]", cigar))
if (length(m) > 0) left_clip <- as.integer(sub("[SH]", "", m))

# Soft-clip droit (S) ou hard-clip droit (H)
right_clip <- 0
m <- regmatches(cigar, regexpr("([0-9]+)[SH]$", cigar))
if (length(m) > 0) right_clip <- as.integer(sub("[SH]", "", m))
for (i in seq_len(nrow(mm_somatic))) {
    
    # Position dans le contig complet (pattern(best_aln) commence après le soft-clip)
    pos_in_contig <- mm_somatic$PatternStart[i]
    
    # Décalage : pos SAM = position génomique de la première base MAPPÉE
    # donc on soustrait le soft-clip pour avoir la position de la base 1 du contig
    pos_mut_genome <- if (is_reverse) {
        pos + nchar(as.character(contig)) - right_clip - pos_in_contig
    } else {
        pos - left_clip + pos_in_contig - 1
    }
    
    snv_ref <- as.character(mm_somatic$PatternSubstring[i])
    snv_alt <- as.character(mm_somatic$SubjectSubstring[i])
    
    if (is_reverse) {
        snv_ref <- as.character(reverseComplement(DNAString(snv_ref)))
        snv_alt <- as.character(reverseComplement(DNAString(snv_alt)))
    }
    
    cat("Chromosome :", as.character(chrom),
        "| Position :", pos_mut_genome,
        "| SNV :", snv_ref, "->", snv_alt, "\n")
}


[1] TRUE

Chromosome : 21 | Position : 10896818 | SNV : G -> C 
Chromosome : 21 | Position : 10896817 | SNV : A -> G 


## Pipeline for multiple SNVs

In [9]:
cat("Chargement des reads tumeur (tumeur specifique)...\n")
fq_tumour_specific_R1 <- readDNAStringSet("/srv/home/mlef0011/VDARK/rawdata/reads/tumour_R1_tumour_specific.fq", format="fastq")
fq_tumour_specific_R2 <- readDNAStringSet("/srv/home/mlef0011/VDARK/rawdata/reads/tumour_R2_tumour_specific.fq", format="fastq")
tumour_specific_reads_df <- data.frame(
    read_ID  = c(names(fq_tumour_specific_R1), names(fq_tumour_specific_R2)),
    sequence = as.character(c(fq_tumour_specific_R1, fq_tumour_specific_R2)),
    stringsAsFactors = FALSE
)
cat(nrow(tumour_specific_reads_df), "reads tumeur\n")

Chargement des reads tumeur (tumeur specifique)...
52644 reads tumeur


In [37]:
system("/srv/home/mlef0011/anaconda3/condabin/conda run -n VDARK python3 /srv/home/mlef0011/VDARK/src/build_read_kmer_table.py")

In [11]:
# ── Clustering ─────────────────────────────────────────────────────────────────
cat("Read clustering...\n")
read_kmer_df <- fread("/srv/home/mlef0011/VDARK/rawdata/reads/read_kmer_association.tsv")
read_kmer_df <- read_kmer_df[, if (.N <= 50) .SD, by = kmer]  # retirer k-mers trop fréquents

pairs_weighted <- read_kmer_df[, {
    ids <- read_ID
    if (length(ids) >= 2) {
        idx <- combn(length(ids), 2)
        data.table(read_ID.x = ids[idx[1,]], read_ID.y = ids[idx[2,]])
    }
}, by = kmer][, .(weight = .N), by = .(read_ID.x, read_ID.y)]

G <- graph_from_data_frame(pairs_weighted, directed = FALSE)
E(G)$weight <- pairs_weighted$weight
clusters <- components(G)
cat(clusters$no, "clusters\n")

Read clustering...
3235 clusters


In [13]:
SNV_list <- lapply(c(1:5), function(cluster_ID) {

    message("\n========== Cluster ", cluster_ID, " ==========")

    # Get reads and k-mers from cluster
    reads_in_cluster <- names(clusters$membership)[clusters$membership == cluster_ID]

    kmers_in_cluster <- unique(read_kmer_df$kmer[
        read_kmer_df$read_ID %in% reads_in_cluster
    ])

    all_kmers_cluster <- unique(unlist(lapply(
        tumour_specific_reads_df$sequence[
            tumour_specific_reads_df$read_ID %in% reads_in_cluster
        ],
        get_kmers
    )))

    message(length(reads_in_cluster), " reads | ",
            length(kmers_in_cluster), " specific k-mers")

    # Assemble tumour contig
    contig_tumour <- assemble_kmers(kmers_in_cluster, k = K)

    message("Tumour contig: ", nchar(contig_tumour[1]), " bp")


    # ── Extract flanking kmers ─────────────────────────────

    flanking_kmers <- setdiff(all_kmers_cluster, kmers_in_cluster)

    flanking_fa <- paste0("/srv/home/mlef0011/VDARK/tmp/flanking_kmers_", cluster_ID, ".fa")

    writeLines(
        paste0(">kmer_", seq_along(flanking_kmers), "\n", flanking_kmers),
        flanking_fa
    )


    # Build KMC database
    system(paste(
        "/srv/home/mlef0011/VDARK/software/kmc/bin/kmc -k31 -t12 -ci1 -fm",
        flanking_fa,
        paste0("/srv/home/mlef0011/VDARK/tmp/flanking_kmc_", cluster_ID),
        "/srv/home/mlef0011/VDARK/tmp/"
    ))


    # Rough filtering of normal reads
    system(paste(
        "/srv/home/mlef0011/VDARK/software/kmc/bin/kmc_tools filter",
        paste0("/srv/home/mlef0011/VDARK/tmp/flanking_kmc_", cluster_ID), "-ci1",
        "/srv/home/mlef0011/VDARK/rawdata/normal_R1.fq",
        paste0("/srv/home/mlef0011/VDARK/tmp/normal_R1_locus_", cluster_ID, ".fq")
    ))

    system(paste(
        "/srv/home/mlef0011/VDARK/software/kmc/bin/kmc_tools filter",
        paste0("/srv/home/mlef0011/VDARK/tmp/flanking_kmc_", cluster_ID), "-ci1",
        "/srv/home/mlef0011/VDARK/rawdata/normal_R2.fq",
        paste0("/srv/home/mlef0011/VDARK/tmp/normal_R2_locus_", cluster_ID, ".fq")
    ))


    # Load filtered reads
    fq_R1 <- readDNAStringSet(
        paste0("/srv/home/mlef0011/VDARK/tmp/normal_R1_locus_", cluster_ID, ".fq"),
        format = "fastq"
    )

    fq_R2 <- readDNAStringSet(
        paste0("/srv/home/mlef0011/VDARK/tmp/normal_R2_locus_", cluster_ID, ".fq"),
        format = "fastq"
    )

    normal_df <- data.frame(
        read_ID  = c(names(fq_R1), names(fq_R2)),
        sequence = as.character(c(fq_R1, fq_R2)),
        stringsAsFactors = FALSE
    )


    # Precise filtering using flanking k-mers
    pdict <- PDict(unique(c(
        flanking_kmers,
        as.character(reverseComplement(DNAStringSet(flanking_kmers)))
    )))

    n_hits <- vapply(
        vwhichPDict(pdict, DNAStringSet(normal_df$sequence)),
        length,
        integer(1)
    )

    normal_reads <- normal_df$read_ID[n_hits >= 30]
    
    if (length(normal_reads) == 0) {
        message("No normal reads matching flanking kmers")
        return(NULL)
    }
    message(length(normal_reads), " normal reads retained")


    # Assemble normal contig
    all_kmers_normal <- unlist(lapply(
        normal_df$sequence[normal_df$read_ID %in% normal_reads],
        get_kmers
    ))

    kmer_counts <- table(all_kmers_normal)

    contig_normal <- assemble_kmers(
        names(kmer_counts[kmer_counts >= 10]),
        k = K
    )

    message("Normal contig: ", nchar(contig_normal[1]), " bp")


    # ── Align tumour vs normal ─────────────────────────────

    orientations <- list(
        c(contig_normal[1], contig_tumour[1]),
        c(contig_normal[1], contig_tumour[2]),
        c(contig_normal[2], contig_tumour[1]),
        c(contig_normal[2], contig_tumour[2])
    )

    alns <- lapply(orientations, function(x) {
        pairwiseAlignment(
            DNAString(x[1]),
            DNAString(x[2]),
            type = "local",
            substitutionMatrix = nucleotideSubstitutionMatrix(match = 1, mismatch = -1),
            gapOpening = -5,
            gapExtension = -2
        )
    })

    best_aln <- alns[[which.max(sapply(alns, score))]]


    # Display alignment
    pat <- strsplit(as.character(pattern(best_aln)), "")[[1]]
    sub <- strsplit(as.character(subject(best_aln)), "")[[1]]

    diff <- ifelse(pat == sub, ".", "*")

    cat("Normal :", paste(pat, collapse=""), "\n")
    cat("        ", paste(diff, collapse=""), "\n")
    cat("Tumour :", paste(sub, collapse=""), "\n")


    # ── Detect SNVs ─────────────────────────────

    mm <- mismatchTable(best_aln)

    if (nrow(mm) == 0) {
        message("No mutations detected")
        return(NULL)
    }

    germline <- is_germline_fast(mm, contig_normal)

    mm_somatic <- mm[!germline, ]

    if (nrow(mm_somatic) == 0) {
        message("Only germline variants detected")
        return(NULL)
    }

    message(nrow(mm_somatic), " somatic mutation(s) detected")

    return(list(
        cluster_ID = cluster_ID,
        mismatches = mm_somatic,
        contig_tumour = contig_tumour,
        contig_normal = contig_normal
    ))

})

SNV_list <- Filter(Negate(is.null), SNV_list)

message("\n", length(SNV_list), " cluster(s) with SNVs detected\n")


========== Cluster 1 ==========

20 reads | 53 specific k-mers

Tumour contig: 61 bp

52 normal reads retained

Normal contig: 122 bp



Normal : AGCTCTCTAAGACTGCACTACAACCTGTGTGATCCCTATTGCTTCCAGAAATATAA 
         ..............................*......................... 
Tumour : AGCTCTCTAAGACTGCACTACAACCTGTGTAATCCCTATTGCTTCCAGAAATATAA 


1 somatic mutation(s) detected


========== Cluster 2 ==========

18 reads | 31 specific k-mers

Tumour contig: 61 bp

82 normal reads retained

Normal contig: 232 bp



Normal : GGTCCAGTGGCTTCAGATCCCTGGGTTATGGAGGCTGTGGCTTCCCTTCCCTGGGCTATGG 
         ..............................*.............................. 
Tumour : GGTCCAGTGGCTTCAGATCCCTGGGTTATGAAGGCTGTGGCTTCCCTTCCCTGGGCTATGG 


1 somatic mutation(s) detected


========== Cluster 3 ==========

5 reads | 2 specific k-mers

Tumour contig: 32 bp

No normal reads matching flanking kmers


========== Cluster 4 ==========

8 reads | 31 specific k-mers

Tumour contig: 61 bp

31 normal reads retained

Normal contig: 143 bp



Normal : ACACACACACACCCCTCAACACACCATGCCCTTTGTCTCTCTTTGTATTCCTCTGGCCAGG 
         ..............................*.............................. 
Tumour : ACACACACACACCCCTCAACACACCATGCCGTTTGTCTCTCTTTGTATTCCTCTGGCCAGG 


1 somatic mutation(s) detected


========== Cluster 5 ==========

17 reads | 31 specific k-mers

Tumour contig: 61 bp

59 normal reads retained

Normal contig: 234 bp



Normal : GACAGGACCCTTCTGATGGCAACGTTGGCTGAAGGACCACATGGCATTCTTGTGGAACCTT 
         ..............................*.............................. 
Tumour : GACAGGACCCTTCTGATGGCAACGTTGGCTAAAGGACCACATGGCATTCTTGTGGAACCTT 


1 somatic mutation(s) detected


4 cluster(s) with SNVs detected




In [18]:
fasta_contig_normal = "/srv/home/mlef0011/VDARK/rawdata/contigs/contigs_normal.fa"

fasta_lines <- unlist(lapply(SNV_list, function(x) {
    c(paste0(">contig_cluster_", x$cluster_ID),
      x$contig_normal[1])
}))
writeLines(fasta_lines, fasta_contig_normal)

In [19]:
minimap = "/srv/home/mlef0011/anaconda3/envs/VDARK/bin/minimap2"
reference_genome = "/srv/home/mlef0011/rawdata/ref_genome/reference_genome_GRCh37.fa"
alignment_minimap_normal_contig = "/srv/home/mlef0011/VDARK/rawdata/contigs/contigs_normal_aln.sam"
system(paste(minimap,"-ax sr", reference_genome, fasta_contig_normal, ">", alignment_minimap_normal_contig))

In [20]:
# --- Lire le SAM ---
sam <- read.table(alignment_minimap_normal_contig, 
                  sep="\t", comment.char="@", fill=TRUE)
# Flag du read
flag <- sam[1, 2]
is_reverse <- bitwAnd(flag, 16) != 0  # TRUE si brin reverse
is_reverse
chrom <- sam[1, 3]
pos   <- sam[1, 4]  # position de début du contig sur le génome
contig = sam[1, 10]
# --- Boucle sur les mutations détectées ---
# Parser le soft-clip depuis le CIGAR
cigar <- as.character(sam[1, 6])

# Soft-clip gauche (S) ou hard-clip gauche (H)
left_clip <- 0
m <- regmatches(cigar, regexpr("^([0-9]+)[SH]", cigar))
if (length(m) > 0) left_clip <- as.integer(sub("[SH]", "", m))

# Soft-clip droit (S) ou hard-clip droit (H)
right_clip <- 0
m <- regmatches(cigar, regexpr("([0-9]+)[SH]$", cigar))
if (length(m) > 0) right_clip <- as.integer(sub("[SH]", "", m))
for (i in seq_len(nrow(mm_somatic))) {
    
    # Position dans le contig complet (pattern(best_aln) commence après le soft-clip)
    pos_in_contig <- mm_somatic$PatternStart[i]
    
    # Décalage : pos SAM = position génomique de la première base MAPPÉE
    # donc on soustrait le soft-clip pour avoir la position de la base 1 du contig
    pos_mut_genome <- if (is_reverse) {
        pos + nchar(as.character(contig)) - right_clip - pos_in_contig
    } else {
        pos - left_clip + pos_in_contig - 1
    }
    
    snv_ref <- as.character(mm_somatic$PatternSubstring[i])
    snv_alt <- as.character(mm_somatic$SubjectSubstring[i])
    
    if (is_reverse) {
        snv_ref <- as.character(reverseComplement(DNAString(snv_ref)))
        snv_alt <- as.character(reverseComplement(DNAString(snv_alt)))
    }
    
    cat("Chromosome :", as.character(chrom),
        "| Position :", pos_mut_genome,
        "| SNV :", snv_ref, "->", snv_alt, "\n")
}


[1] TRUE

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'nrow': object 'mm_somatic' not found


In [ ]:
sam <- read.table(alignment_minimap_normal_contig, 
                  sep="\t", comment.char="@", fill=TRUE)
# Flag du read
flag <- sam[1, 2]
is_reverse <- bitwAnd(flag, 16) != 0  # TRUE si brin reverse
is_reverse
chrom <- sam[1, 3]
pos   <- sam[1, 4]  # position de début du contig sur le génome
contig = sam[1, 10]
# --- Boucle sur les mutations détectées ---
# Parser le soft-clip depuis le CIGAR
cigar <- as.character(sam[1, 6])

# Soft-clip gauche (S) ou hard-clip gauche (H)
left_clip <- 0
m <- regmatches(cigar, regexpr("^([0-9]+)[SH]", cigar))
if (length(m) > 0) left_clip <- as.integer(sub("[SH]", "", m))

# Soft-clip droit (S) ou hard-clip droit (H)
right_clip <- 0
m <- regmatches(cigar, regexpr("([0-9]+)[SH]$", cigar))
if (length(m) > 0) right_clip <- as.integer(sub("[SH]", "", m))
for (i in seq_len(nrow(mm_somatic))) {
    
    # Position dans le contig complet (pattern(best_aln) commence après le soft-clip)
    pos_in_contig <- mm_somatic$PatternStart[i]
    
    # Décalage : pos SAM = position génomique de la première base MAPPÉE
    # donc on soustrait le soft-clip pour avoir la position de la base 1 du contig
    pos_mut_genome <- if (is_reverse) {
        pos + nchar(as.character(contig)) - right_clip - pos_in_contig
    } else {
        pos - left_clip + pos_in_contig - 1
    }
    
    snv_ref <- as.character(mm_somatic$PatternSubstring[i])
    snv_alt <- as.character(mm_somatic$SubjectSubstring[i])
    
    if (is_reverse) {
        snv_ref <- as.character(reverseComplement(DNAString(snv_ref)))
        snv_alt <- as.character(reverseComplement(DNAString(snv_alt)))
    }
    
    cat("Chromosome :", as.character(chrom),
        "| Position :", pos_mut_genome,
        "| SNV :", snv_ref, "->", snv_alt, "\n")
}

In [22]:
# --- Lire le SAM ---
sam <- read.table("/srv/home/mlef0011/VDARK/rawdata/contigs/contigs_normal_aln.sam", 
                  sep="\t", comment.char="@", fill=TRUE)


for(x in seq_len(length(SNV_list))){
    cat("------------------------\n")
    mm = SNV_list[[x]]$mismatches
    flag <- sam[x, 2]
    is_reverse <- bitwAnd(flag, 16) != 0  # TRUE si brin reverse
    is_reverse
    chrom <- sam[x, 3]
    pos   <- sam[x, 4]  # position de début du contig sur le génome
    contig = sam[x, 10]
    # --- Boucle sur les mutations détectées ---
    # Parser le soft-clip depuis le CIGAR
    cigar <- as.character(sam[x, 6])

    # Soft-clip gauche (S) ou hard-clip gauche (H)
    left_clip <- 0
    m <- regmatches(cigar, regexpr("^([0-9]+)[SH]", cigar))
    if (length(m) > 0) left_clip <- as.integer(sub("[SH]", "", m))

    # Soft-clip droit (S) ou hard-clip droit (H)
    right_clip <- 0
    m <- regmatches(cigar, regexpr("([0-9]+)[SH]$", cigar))
    if (length(m) > 0) right_clip <- as.integer(sub("[SH]", "", m))

    # --- Boucle sur les mutations détectées ---
    for (i in seq_len(nrow(mm))) {

        # Calcul de la position génomique
        pos_mut_genome <- if (is_reverse) {        
            pos + nchar(contig) - mm$PatternStart[i]
        } else {
            pos + mm$PatternStart[i] - 1
        }

        # SNV sur le brin reverse si nécessaire
        snv_ref <- as.character(mm$PatternSubstring[i])
        snv_alt <- as.character(mm$SubjectSubstring[i])

        if (is_reverse) {
            snv_ref <- as.character(reverseComplement(DNAString(snv_ref)))
            snv_alt <- as.character(reverseComplement(DNAString(snv_alt)))
        }

        # Affichage
        cat("Chromosome :", as.character(chrom), 
            "| Position :", pos_mut_genome,
            "| SNV :", snv_ref, "->", snv_alt, "\n")
    }
}

------------------------
Chromosome : 21 | Position : 31442822 | SNV : C -> T 
------------------------
Chromosome : 21 | Position : 31768802 | SNV : G -> A 
------------------------
Chromosome : 21 | Position : 44593920 | SNV : C -> G 
------------------------
Chromosome : 21 | Position : 33221825 | SNV : C -> T 


In [36]:
mutations_data = fread("/srv/home/mlef0011/VDARK/rawdata/448fe471-3f4e-4dc8-a4e0-6f147dc93abe.consensus.20160830.somatic.snv_mnv.vcf.gz", header=T, sep = "\t")
mutations_data[mutations_data$`#CHROM`== 21 & mutations_data$POS == 31442822 ,]
mutations_data[mutations_data$`#CHROM`== 21 & mutations_data$POS == 31768802 ,]
mutations_data[mutations_data$`#CHROM`== 21 & mutations_data$POS == 33221825 ,]
mutations_data[mutations_data$`#CHROM`== 21 & mutations_data$POS == 44593920 ,]


#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
21,31442822,.,C,T,.,.,"Callers=broad,dkfz,muse;NumCallers=3;VAF=0.35;t_alt_count=14;t_ref_count=26;Variant_Classification=IGR"


#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
21,31768802,.,G,A,.,.,"Callers=broad,dkfz,muse,sanger;NumCallers=4;VAF=0.6667;cosmic=COSM419269;t_alt_count=18;t_ref_count=9;Variant_Classification=Missense_Mutation"


#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
21,33221825,.,C,T,.,.,"Callers=broad,dkfz,muse,sanger;NumCallers=4;VAF=0.5385;dbsnp=rs776300076;t_alt_count=14;t_ref_count=12;Variant_Classification=IGR"


#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
21,44593920,.,C,G,.,.,"Callers=broad,muse,sanger;NumCallers=3;VAF=0.4706;t_alt_count=8;t_ref_count=9;Variant_Classification=IGR"


In [29]:
contig_sam = "/srv/home/mlef0011/VDARK/rawdata/contigs/contigs_normal_aln.sam"

In [30]:
sam <- read.table(contig_sam, sep = "\t", comment.char = "@", fill = TRUE)


In [39]:
snv_rows <- lapply(seq_along(SNV_list), function(i) {
    x    <- SNV_list[[i]]
    mm   <- x$mismatches
    flag <- as.integer(sam[i, 2])

    if (is.na(flag) || bitwAnd(flag, 4) != 0 || bitwAnd(flag, 2048) != 0) {
        log_msg("  Cluster ", x$cluster_ID, " | contig unmapped or supplementary -- skipping")
        return(NULL)
    }

    is_reverse <- bitwAnd(flag, 16) != 0
    chrom      <- as.character(sam[i, 3])
    pos        <- as.integer(sam[i, 4])
    contig_len <- nchar(as.character(sam[i, 10]))

    cigar     <- as.character(sam[i, 6])
    m         <- regmatches(cigar, regexpr("^([0-9]+)[SH]", cigar))
    left_clip <- if (length(m) > 0) as.integer(sub("[SH]", "", m)) else 0

    lapply(seq_len(nrow(mm)), function(j) {
        mut_pos <- if (is_reverse) {
            pos + contig_len - mm$PatternStart[j]
        } else {
            pos + mm$PatternStart[j] - 1L
        }
        ref_nt <- as.character(mm$PatternSubstring[j])
        alt_nt <- as.character(mm$SubjectSubstring[j])
        if (is_reverse) {
            ref_nt <- as.character(reverseComplement(DNAString(ref_nt)))
            alt_nt <- as.character(reverseComplement(DNAString(alt_nt)))
        }
        data.frame(CHROM = chrom, POS = mut_pos, REF = ref_nt, ALT = alt_nt,
                   stringsAsFactors = FALSE)
    })
})

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'X' in selecting a method for function 'lapply': object 'SNV_list' not found
